In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [ ]:
df = pd.read_csv('../data/dataset.csv', index_col=0)
df.head()

## EDA — Limpieza

In [ ]:
print(df.isnull().sum())
df = df.dropna(subset=['artists', 'album_name', 'track_name'])
df = df.drop_duplicates(subset=['track_id'])
print(df.shape)

In [ ]:
features = ['popularity','danceability','energy','loudness','speechiness',
            'acousticness','instrumentalness','liveness','valence','tempo']

print(df[features].describe())

## EDA — Outliers en popularity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['popularity'].hist(bins=50, ax=axes[0])
axes[0].set_title('Distribución de popularity (original)')

Q1 = df['popularity'].quantile(0.25)
Q3 = df['popularity'].quantile(0.75)
IQR = Q3 - Q1
df_clean = df[(df['popularity'] >= Q1 - 1.5*IQR) & (df['popularity'] <= Q3 + 1.5*IQR)]

df_clean['popularity'].hist(bins=50, ax=axes[1])
axes[1].set_title('Distribución de popularity (sin outliers)')

plt.tight_layout()
plt.show()
print('Correlacion antes:', df[features].corr()['popularity'].to_dict())
print('Correlacion despues:', df_clean[features].corr()['popularity'].to_dict())

## PCA sobre features de audio

In [ ]:
audio_features = ['danceability','energy','loudness','speechiness',
                  'acousticness','instrumentalness','liveness','valence','tempo']

X = df_clean[audio_features].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=9)
X_pca = pca.fit_transform(X_scaled)

loadings = pd.DataFrame(pca.components_.T,
                        index=audio_features,
                        columns=[f'PC{i+1}' for i in range(9)])
print(loadings.round(4))

In [ ]:
threshold = 0.3
for col in loadings.columns:
    important = loadings[col][loadings[col].abs() >= threshold].index.tolist()
    print(f'{col}: {important}')

In [ ]:
explained = pca.explained_variance_ratio_
plt.figure(figsize=(8, 4))
plt.bar(range(1, 10), explained)
plt.plot(range(1, 10), np.cumsum(explained), marker='o', color='red', label='Acumulado')
plt.xlabel('Componente')
plt.ylabel('Varianza explicada')
plt.title('Varianza explicada por componente')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
sample = df_clean.copy()
sample = sample[sample['track_genre'].isin(sample['track_genre'].value_counts().head(10).index)]
X_s = scaler.transform(sample[audio_features].values)
coords = pca.transform(X_s)

genres = sample['track_genre'].unique()
colors = plt.cm.tab10(np.linspace(0, 1, len(genres)))

plt.figure(figsize=(10, 6))
for g, c in zip(genres, colors):
    mask = sample['track_genre'] == g
    plt.scatter(coords[mask, 0], coords[mask, 1], label=g, alpha=0.4, s=10, color=c)

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA — PC1 vs PC2 por género')
plt.legend(markerscale=3, fontsize=8)
plt.tight_layout()
plt.show()

## MDS

In [ ]:
from sklearn.manifold import MDS

sample_mds = df_clean.groupby('track_genre').apply(lambda x: x.sample(min(50, len(x)), random_state=42)).reset_index(drop=True)
X_mds = scaler.fit_transform(sample_mds[audio_features].values)

mds = MDS(n_components=2, random_state=42, max_iter=300)
coords_mds = mds.fit_transform(X_mds)

top_genres = sample_mds['track_genre'].value_counts().head(10).index
mask_top = sample_mds['track_genre'].isin(top_genres)

plt.figure(figsize=(10, 6))
for g, c in zip(top_genres, plt.cm.tab10(np.linspace(0, 1, 10))):
    m = (sample_mds['track_genre'] == g).values
    plt.scatter(coords_mds[m, 0], coords_mds[m, 1], label=g, alpha=0.5, s=15, color=c)

plt.xlabel('MDS 1')
plt.ylabel('MDS 2')
plt.title('MDS — similitud entre géneros')
plt.legend(markerscale=3, fontsize=8)
plt.tight_layout()
plt.show()

## Distancia de edición entre géneros

In [ ]:
def edit_distance(s1, s2):
    m, n = len(s1), len(s2)
    L = np.zeros((m+1, n+1), dtype=int)
    for i in range(m+1):
        L[i, 0] = i
    for j in range(n+1):
        L[0, j] = j
    for i in range(1, m+1):
        for j in range(1, n+1):
            f = 0 if s1[i-1] == s2[j-1] else 1
            L[i, j] = min(L[i-1, j]+1, L[i, j-1]+1, L[i-1, j-1]+f)
    return L, L[m, n]

genres_sample = ['acoustic', 'alt-rock', 'alternative', 'blues', 'bluegrass']

matrix = pd.DataFrame(index=genres_sample, columns=genres_sample)
for g1 in genres_sample:
    for g2 in genres_sample:
        _, d = edit_distance(g1, g2)
        matrix.loc[g1, g2] = d

print(matrix)

In [ ]:
L, dist = edit_distance('acoustic', 'alt-rock')
df_matrix = pd.DataFrame(L,
    index=['-'] + list('acoustic'),
    columns=['-'] + list('alt-rock'))
print(f"Distancia entre 'acoustic' y 'alt-rock': {dist}")
print(df_matrix)